In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import warnings

def generate_station_in(shp_filepath, out_filepath="station.in"):
    print(f"Reading shapefile: {shp_filepath}")
    gdf = gpd.read_file(shp_filepath)
    
    # --- THE FIX IS HERE ---
    # 1. If CRS is completely missing, assume it's already lon/lat (WGS84) and set it
    if gdf.crs is None:
        print("Warning: No CRS detected (likely missing .prj file). Assuming data is already WGS84 (EPSG:4326)...")
        gdf.set_crs(epsg=4326, inplace=True)
    # 2. If it has a CRS but it's not WGS84, convert it
    elif gdf.crs.to_epsg() != 4326:
        print("Converting coordinate reference system to WGS84...")
        gdf = gdf.to_crs(epsg=4326)
    # -----------------------
        
    station_lines = []
    station_id = 1
    
    # Ignore geopandas warnings about centroids on geographic CRS
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        
        for index, row in gdf.iterrows():
            lon = row.geometry.centroid.x
            lat = row.geometry.centroid.y
            
            # Convert longitude to 0-360 format (for the Bering Sea)
            if lon < 0:
                lon += 360.0
                
            # Determine depth profile based on the 'source' attribute
            source_upper = str(row['source']).strip().upper()
            
            if source_upper in ['CO-OPS', 'NDBC']:
                depths = [0.0, -1.0]
            else:
                depths = np.arange(0.0, -100.0, -2.0)
                
            # Fetch attributes (filling with empty string if missing/nan)
            var_str = row['var'] if not pd.isna(row['var']) else ""
            code_str = row['code'] if not pd.isna(row['code']) else ""
            src_str = row['source'] if not pd.isna(row['source']) else ""
            name_str = row['name'] if not pd.isna(row['name']) else ""
            
            for d in depths:
                z_comment = f"z={d:g}m" 
                line = f"{station_id} {lon:.4f} {lat:.4f} {d:.1f} ! [{var_str}],{code_str},{src_str},{name_str} {z_comment}"
                station_lines.append(line)
                station_id += 1
                
    total_stations = len(station_lines)
    
    # Write everything to the station.in file
    print(f"Writing {total_stations} station points to {out_filepath}...")
    with open(out_filepath, 'w') as f:
        f.write("1 1 1 1 1 1 1 1 1 !on (1)|off(0) flags for elev,air pressure,windx,windy,T,S,u,v,w,rest of tracers (expanded into subclasses of each module)\n")
        f.write(f"{total_stations} !# of stations\n")
        
        for line in station_lines:
            f.write(line + "\n")
            
    print("Process complete!")

if __name__ == "__main__":
    INPUT_SHP = "./stations.shp" 
    generate_station_in(INPUT_SHP, out_filepath="station.in")

Reading shapefile: ./stations.shp
Writing 486 station points to station.in...
Process complete!
